# 01 - Llamada LLM transparente

Este notebook enseña la llamada LLM desde el cableado manual. Primero verás `system_prompt`, `user_prompt` y `messages`, igual que en una llamada directa a OpenAI. Después aparece el toolkit como una forma de no repetir ese cableado.

## Regla de este toolkit

Antes de usar una abstracción, debemos poder responder: ¿qué se envió exactamente al modelo?

Por eso este notebook muestra tres niveles:

1. Manual: strings y `messages` explícitos.
2. Helper pequeño: `build_messages(system, user)`.
3. Toolkit: `LLMFactory`, `llm.invoke(...)` y `PromptRegistry`.

In [1]:
from llmkit.config.settings import get_settings
from llmkit.llms import LLMFactory, build_messages
from llmkit.prompts import PromptRegistry

get_settings.cache_clear()
settings = get_settings()

print("Default provider:", settings.default_provider)
print("Default model:", settings.default_model)
print("Default temperature:", settings.temperature)

Default provider: openai
Default model: gpt-5-nano
Default temperature: 0.2


## 1. Variables manuales

`MODEL_ID` decide qué modelo se usará. Si contiene solo `gpt-5-nano`, el toolkit usa `settings.default_provider`, normalmente `openai`. También puedes escribir `openai:gpt-5-nano` para ser explícito.

`topic` es el dato de negocio que cambia en cada llamada.

`system_prompt` afecta el comportamiento del modelo: rol, tono, límites y forma de responder.

`user_prompt` contiene la tarea concreta para esta llamada.

In [2]:
MODEL_ID = settings.default_model
topic = "virtual sensors in paper processes"

system_prompt = "You are a concise technical assistant."
user_prompt = f"Explain the following topic in practical terms: {topic}"

print("MODEL_ID:", MODEL_ID)
print("\nSYSTEM PROMPT:\n", system_prompt)
print("\nUSER PROMPT:\n", user_prompt)

MODEL_ID: gpt-5-nano

SYSTEM PROMPT:
 You are a concise technical assistant.

USER PROMPT:
 Explain the following topic in practical terms: virtual sensors in paper processes


## 2. Lo que se envía al modelo

Esto es lo que Ed arma manualmente con `messages=[...]`. No hay magia: la llamada chat usa una lista de mensajes con roles.

In [3]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

messages

[{'role': 'system', 'content': 'You are a concise technical assistant.'},
 {'role': 'user',
  'content': 'Explain the following topic in practical terms: virtual sensors in paper processes'}]

## 3. Helper equivalente

`build_messages(system, user)` no decide nada. Solo construye la misma lista anterior. Existe para que clientes OpenAI, Ollama y compatibles usen la misma forma.

In [4]:
helper_messages = build_messages(system_prompt, user_prompt)

assert helper_messages == messages
helper_messages

[{'role': 'system', 'content': 'You are a concise technical assistant.'},
 {'role': 'user',
  'content': 'Explain the following topic in practical terms: virtual sensors in paper processes'}]

## 4. Llamada con el toolkit

`LLMFactory.create(MODEL_ID)` solo selecciona el cliente correcto. No decide el prompt.

`llm.invoke(system=..., user=...)` recibe los dos strings explícitos. Internamente usa el mismo formato `messages` que acabamos de imprimir.

La llamada real está protegida por `RUN_LIVE_LLM_CALLS` para evitar costos accidentales.

In [5]:
RUN_LIVE_LLM_CALLS = False

if RUN_LIVE_LLM_CALLS:
    llm = LLMFactory.create(MODEL_ID)
    response = llm.invoke(system=system_prompt, user=user_prompt)
    print(response.content)
else:
    print("Demo mode: no se llamó al modelo.")
    print("Si RUN_LIVE_LLM_CALLS=True, se enviaría este messages:")
    print(messages)

Demo mode: no se llamó al modelo.
Si RUN_LIVE_LLM_CALLS=True, se enviaría este messages:
[{'role': 'system', 'content': 'You are a concise technical assistant.'}, {'role': 'user', 'content': 'Explain the following topic in practical terms: virtual sensors in paper processes'}]


## 5. Mismo prompt guardado en PromptRegistry

Ahora que ya vimos el prompt manual, podemos guardarlo en el registry. El registry no cambia la llamada; solo evita copiar y pegar strings.

`prompt.system` equivale a `system_prompt`.

`prompt.user_template` equivale a una plantilla del `user_prompt`.

`prompt.render_user(topic=...)` rellena la plantilla y valida que no falte `topic`.

In [6]:
prompt = PromptRegistry.get("chat.basic")
registry_user_prompt = prompt.render_user(topic=topic)
registry_messages = build_messages(prompt.system, registry_user_prompt)

print("Prompt name:", prompt.name)
print("Description:", prompt.description)
print("Required variables:", prompt.required_variables)
print("\nSYSTEM FROM REGISTRY:\n", prompt.system)
print("\nUSER FROM REGISTRY:\n", registry_user_prompt)
print("\nMESSAGES FROM REGISTRY:")
registry_messages

Prompt name: chat.basic
Description: Basic technical explanation prompt.
Required variables: ['topic']

SYSTEM FROM REGISTRY:
 You are a concise technical assistant.

USER FROM REGISTRY:
 Explain the following topic in practical terms: virtual sensors in paper processes

MESSAGES FROM REGISTRY:


[{'role': 'system', 'content': 'You are a concise technical assistant.'},
 {'role': 'user',
  'content': 'Explain the following topic in practical terms: virtual sensors in paper processes'}]

## Lectura mental correcta

Cuando veas esto:

```python
llm.invoke(system=prompt.system, user=prompt.render_user(topic=topic))
```

léelo como esto:

```python
messages = [
    {"role": "system", "content": prompt.system},
    {"role": "user", "content": prompt.render_user(topic=topic)},
]
```

Esa equivalencia es la base del toolkit.